## Load the regions bed files and perform the enrichment analysis vs remap2022

In [2]:
library(dplyr)
library(ggplot2)
library(regioneReloaded)
library(BSgenome.Hsapiens.UCSC.hg38)


### NGN2 focus: undiffWTC11 dCREs are not filtered out + very small subset as background set

In [3]:
# background sequences: (why 0.01? Stringent definition of around zero activity; Why 0.04 a bigger set of sequences without statistical significant activity different to scrambled sequences. looked for a threshold which uses the most regions but still has no overlap with the considered active regions)

input_path = "/home/kisa/coding/80K_MPRA/remap_chipseq_pia" # no tailing slash allowed
output_directory <- "/home/kisa/coding/80K_MPRA/remap_chipseq_pia/80k_results" # no trailing slash allowed

around_zero_activity_both_cell_types_001 <- file.path(output_directory, "80k_results/around_zero_activity_both_cell_types_2_duplicated_regions_not_stand_aware.bed")

# less conservative background sequences
around_zero_activity_both_cell_types_004 <- file.path(output_directory, "80k_results/around_zero_activity_both_cell_types_2_duplicated_regions_not_stand_aware_Thresh_004.bed")

# NGN2 focus
upbed <- file.path(output_directory, "80k_results/ngn2_increasing_regions_duplicates_without_considering_strand.bed")
downbed <- file.path(output_directory, "80k_results/ngn2_decreasing_regions_duplicates_without_considering_strand.bed")

# remap2022:
remap2022 <- file.path(output_directory, "remap2022_nr_macs2_hg38_v1_0.bed.gz")
# zcat remap2022_nr_macs2_hg38_v1_0.bed.gz | head -n 100000 | gzip >remap2022_nr_macs2_hg38_v1_0_subset_100k.bed.gz
remap2022 <- file.path(output_directory, "remap2022_nr_macs2_hg38_v1_0_subset_100k.bed.gz")
remap2022 <- file.path(output_directory, "remap2022_nr_macs2_hg38_v1_0_subset_10k.bed.gz")
# 68.655.741 regions


In [ ]:
data_up <- read.table(upbed, header = F, sep = "\t", col.names = c("chr", "start", "end", "name", "strand", "score"))
data_down <- read.table(downbed, header = F, sep = "\t", col.names = c("chr", "start", "end", "name", "strand", "score"))

In [14]:
head(data_up)

,chr,start,end,name,strand,score
,<chr>,<int>,<int>,<chr>,<chr>,<int>
1,chrX,148610380,148610650,cardiac_neuro_cava_random:AFF2|ENSG00000155966.14|EH38E2772293_fwd_tile1-1,+,1
2,chrX,148714069,148714339,cardiac_neuro_cava_random:AFF2|ENSG00000155966.14|EH38E2772336_fwd_tile1-1,+,1
3,chrX,148541408,148541678,cardiac_neuro_cava_random:AFF2|ENSG00000155966.14|EH38E3947484_fwd_tile1-1,+,1
4,chr1,27577161,27577431,cardiac_neuro_cava_random:AHDC1|ENSG00000126705.15|EH38E2797930_rev_tile1-1,-,1
5,chr1,27593644,27593914,cardiac_neuro_cava_random:AHDC1|ENSG00000126705.15|EH38E2797946_rev_tile1-1,-,1
6,chr4,113083387,113083657,cardiac_neuro_cava_random:ANK2|ENSG00000145362.21|EH38E2321059_fwd_tile1-1,+,1


In [ ]:
ngn2_increasing_activity_regions <- GenomicRanges::makeGRangesFromDataFrame(data_up, seqnames.field = "chr",start.field = "start",end.field = "end")
ngn2_decreasing_activity_regions <- GenomicRanges::makeGRangesFromDataFrame(data_down, seqnames.field = "chr",start.field = "start",end.field = "end")
ngn2_dCREs <- list(Neuron_upregulating_dCREs = ngn2_increasing_activity_regions, Neuron_downregulating_dCREs = ngn2_decreasing_activity_regions)

In [ ]:
# background sequences
background_conservative <- read.table(around_zero_activity_both_cell_types_001, header = F, sep = "\t", col.names = c("chr", "start", "end", "name", "strand", "score"))
background_not_conservative <- read.table(around_zero_activity_both_cell_types_004, header = F, sep = "\t", col.names = c("chr", "start", "end", "name", "strand", "score"))
both_cell_types_around_zero_activity_001 <- GenomicRanges::makeGRangesFromDataFrame(background_conservative, seqnames.field = "chr", start.field = "start", end.field = "end")
both_cell_types_around_zero_activity_004 <- GenomicRanges::makeGRangesFromDataFrame(background_not_conservative, seqnames.field = "chr", start.field = "start", end.field = "end")


In [ ]:
ngn2_dCREs_and_background <- list(Neuron_upregulating_dCREs = ngn2_increasing_activity_regions, Neuron_downregulating_dCREs = ngn2_decreasing_activity_regions, background_conservative = both_cell_types_around_zero_activity_001, background_not_conservative = both_cell_types_around_zero_activity_004)

In [20]:
head(both_cell_types_around_zero_activity_004)

GRanges object with 6 ranges and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr16 70323127-70323397      -
  [2]    chr16 70325057-70325327      -
  [3]    chr16 70213614-70213884      -
  [4]    chr16 70292655-70292925      -
  [5]    chr12 21786421-21786691      -
  [6]    chr12 21888341-21888611      -
  -------
  seqinfo: 23 sequences from an unspecified genome; no seqlengths

In [ ]:
remap_df <- read.table(remap2022)
TFs <- GenomicRanges::makeGRangesFromDataFrame(remap_df, seqnames.field = "V1", start.field = "V2", end.field = "V3",  keep.extra.columns = T)

In [8]:
head(TFs)

GRanges object with 6 ranges and 6 metadata columns:
      seqnames     ranges strand |            V4        V5          V6
         <Rle>  <IRanges>  <Rle> |   <character> <integer> <character>
  [1]     chr1 9829-10459      * |  SMARCA4:HeLa         1           .
  [2]     chr1 9880-10389      * |  RELB:GM12878         1           .
  [3]     chr1 9883-10270      * | MLLT1:GM12878         1           .
  [4]     chr1 9884-10289      * | ZBTB10:HEK293         1           .
  [5]     chr1 9889-10124      * |   ZFP3:HEK293         1           .
  [6]     chr1 9889-10256      * |   ZNF2:HEK293         1           .
             V7        V8          V9
      <integer> <integer> <character>
  [1]     10084     10085  224,84,252
  [2]     10091     10092  252,168,56
  [3]     10088     10089   140,28,84
  [4]     10106     10107   70,67,181
  [5]     10088     10089  109,44,173
  [6]     10088     10089    1,71,234
  -------
  seqinfo: 1 sequence from an unspecified genome; no seqlengths

Split the remap data by TF name (V4) and save a list of regions for each TF (tflist)

In [ ]:
split_granges <- split(TFs, GenomicRanges::mcols(TFs)$V4)
split_granges <- lapply(split_granges, function(gr) {
    GenomicRanges::mcols(gr)$V4 <- NULL
    return(gr)
})
tflist <- c(split_granges)

In [25]:
# now we performe the crosswise permutation test, using 5000 sampling of the data and 100 permutations.
# The function that we use for randomization if "randomizeRegions" (NOTE: It can take a while, but you
# # can do it in parallel using the argument mc.cores) and the evalutation function is "numOveralps".

# # WITH background sequences
# ngn2_remap_chip_increasing_decreasing <- crosswisePermTest(
#     Alist = ngn2_dCREs_and_background,
#     Blist = tflist,
#     sampling = FALSE,
#     ranFUN = "randomizeRegions",
#     evFUN = "numOverlaps",
#     ntimes = 1000,
#     genome = "hg38",
#     mc.cores = 32
# )
# at least 33 min
# killed after 993 min

# WITHOUT background sequences
ngn2_remap_chip_increasing_decreasing <- crosswisePermTest(
    Alist = ngn2_dCREs,
    Blist = tflist,
    sampling = FALSE,
    ranFUN = "randomizeRegions",
    evFUN = "numOverlaps",
    ntimes = 1000,
    genome = "hg38",
    mc.cores = 32
)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

The masked version of 'hg38' is not installed. Using the unmasked version. This means that no automatic masking will be available.



In [ ]:
# # WITH background sequences
# ngn2_remap_chip_increasing_decreasing_CrosswiseMatrix <- makeCrosswiseMatrix(ngn2_remap_chip_increasing_decreasing, clusterize = TRUE, transform =TRUE) # clusterize = F, transform = T) # , transform=T) #, clusterize=T)

# # WITHOUT background sequences
# # set clusterize to FALSE if only 2 sequence sets
ngn2_remap_chip_increasing_decreasing_CrosswiseMatrix <- makeCrosswiseMatrix(ngn2_remap_chip_increasing_decreasing, clusterize = FALSE, transform =TRUE) # clusterize = F, transform = T) # , transform=T) #, clusterize=T)

output_name <- "ngn2_remap_TF_chip_increasing_decreasing_CrosswiseMatrix.rds"
output_name <- "ngn2_remap_TF_chip_increasing_decreasing_CrosswiseMatrix.rds"

# Use file.path() to combine directory and file name
output_path <- file.path(output_directory, output_name)
saveRDS(ngn2_remap_chip_increasing_decreasing_CrosswiseMatrix, file = output_path)

In [4]:
# # (optional) reading file in
output_name <- "ngn2_remap_TF_chip_increasing_decreasing_CrosswiseMatrix.rds"
output_path <- file.path(output_directory, output_name)

ngn2_remap_chip_increasing_decreasing_CrosswiseMatrix <- readRDS(output_path)

In [5]:
# Extract the two datasets
up_data <- ngn2_remap_chip_increasing_decreasing_CrosswiseMatrix@multiOverlaps$Neuron_upregulating_dCREs %>%
    mutate(regulation = "increasing transcription") # Add a column to label the dataset
down_data <- ngn2_remap_chip_increasing_decreasing_CrosswiseMatrix@multiOverlaps$Neuron_downregulating_dCREs %>%
    mutate(regulation = "decreasing transcription") # Add a column to label the dataset

In [6]:
head(up_data)

,order.id,name,n_regionA,n_regionB,z_score,p_value,n_overlaps,mean_perm_test,sd_perm_test,norm_zscore,adj.p_value,regulation
,<int>,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1,AATF:NALM-6,772,9,-0.0316,0.999001,0,0.001,0.03162278,-0.001137309,1,increasing transcription
2,2,AFF1:K-562,772,1,-0.0316,0.999001,0,0.001,0.03162278,-0.001137309,1,increasing transcription
3,3,AFF1:SEM,772,2,NA,1.000000,0,0.000,0.00000000,NA,1,increasing transcription
4,4,AFF4:HeLa,772,6,-0.0316,0.999001,0,0.001,0.03162278,-0.001137309,1,increasing transcription
5,5,"AFF4:HeLa,CD4",772,2,-0.0316,0.999001,0,0.001,0.03162278,-0.001137309,1,increasing transcription
6,6,AFF4:MCF-7,772,15,-0.0548,0.997003,0,0.003,0.05471740,-0.001972295,1,increasing transcription


In [8]:
# Select top x other enriched experiments (not in specific_experiments) for each dataset
n_top = 15
sig_level = 0.05
top5_up <- up_data %>%
  filter(adj.p_value < sig_level & is.finite(norm_zscore)) %>%
  arrange(desc(norm_zscore)) %>%
  slice_head(n = n_top) %>%
  pull(name)

top5_down <- down_data %>%
  filter(adj.p_value < sig_level & is.finite(norm_zscore)) %>%
  arrange(desc(norm_zscore)) %>%
  slice_head(n = n_top) %>%
  pull(name)

In [14]:
top5_up_non_sig <- up_data %>%
    filter(is.finite(norm_zscore)) %>%
    arrange(desc(norm_zscore)) %>%
    slice_head(n = n_top) %>%
    pull(name)

top5_down_non_sig <- down_data %>%
    filter(is.finite(norm_zscore)) %>%
    arrange(desc(norm_zscore)) %>%
    slice_head(n = n_top) %>%
    pull(name)

In [9]:
# Filter for specific experiments
filtered_up <- up_data %>%
  filter(name %in% c(top5_up))
filtered_down <- down_data %>%
  filter(name %in% c(top5_down))

# Combine all data
combined_data <- bind_rows(filtered_up, filtered_down)

x_limits <- range(c(filtered_up$norm_zscore, filtered_down$norm_zscore), na.rm = TRUE)

Warning message in min(x):
“no non-missing arguments to min; returning Inf”
Warning message in max(x):
“no non-missing arguments to max; returning -Inf”


In [16]:
# Filter for specific experiments
filtered_up <- up_data %>%
    filter(name %in% c(top5_up_non_sig))
filtered_down <- down_data %>%
    filter(name %in% c(top5_down_non_sig))

# Combine all data
combined_data <- bind_rows(filtered_up, filtered_down)

x_limits <- range(c(filtered_up$norm_zscore, filtered_down$norm_zscore), na.rm = TRUE)

In [19]:
# Plot the data
pdf_name_enhancer <-  "dotplot_enhancer_remap2022_TF_enrichment_local.pdf"
pdf(file.path(output_directory, pdf_name_enhancer), width = 8, height = 6) # Adjust width and height
ggplot(filtered_up, aes(x = norm_zscore, y = reorder(name, norm_zscore), fill = regulation)) +
    geom_point(aes(size = -log10(adj.p_value)), shape = 21, alpha = 0.6, stroke = NA) + # Ensure fill is mapped and color is NA
    labs(
        x = "Enrichment (normalized z-score)",
        y = "Experiment",
        title = "Enrichment of CREs"
    ) +
    theme_minimal(base_size = 12) +
    theme(
        legend.position = "top",
        panel.grid.major.y = element_line(color = "gray90"),
        panel.grid.major.x = element_blank(),
        axis.text.y = element_text(size = 10, margin = margin(r = 5)), # Reduce y-axis text size for density
        axis.text.x = element_text(size = 12) # Adjust x-axis text size
    ) +
    scale_fill_manual(
        values = c("Upregulating" = "#33a02c"), # , "Downregulating" = "#e31a1c"),  # Green for Enhancer, Red for Silencing
        labels = c("Upregulating" = "Enhancer"), # , "Downregulating" = "Silencer"),  # Rename labels
        name = "Regulation"
    ) +
    scale_size_continuous(
        range = c(2, 5), # Adjust dot size range for better visibility
        name = "Significance (-log10 p-value)"
    ) +
    guides(
        fill = guide_legend(override.aes = list(size = 6, shape = 21)) # Increase legend dot size
    ) +
    scale_x_continuous(limits = x_limits) +
    coord_cartesian(clip = "off") # Ensure no points are cut off
dev.off()

# # Plot the data
pdf_name_silencer <- "dotplot_silencer_remap2022_TF_enrichment_local.pdf"
pdf(file.path(output_directory, pdf_name_silencer), width = 8, height = 6) # Adjust width and height
ggplot(filtered_down, aes(x = norm_zscore, y = reorder(name, norm_zscore), fill = regulation)) +
    geom_point(aes(size = -log10(adj.p_value)), shape = 21, alpha = 0.6, stroke = NA) + # Ensure fill is mapped and color is NA
    labs(
        x = "Enrichment (normalized z-score)",
        y = "Experiment",
        title = "Enrichment of CREs"
    ) +
    theme_minimal(base_size = 12) +
    theme(
        legend.position = "top",
        panel.grid.major.y = element_line(color = "gray90"),
        panel.grid.major.x = element_blank(),
        axis.text.y = element_text(size = 10, margin = margin(r = 5)), # Reduce y-axis text size for density
        axis.text.x = element_text(size = 12) # Adjust x-axis text size
    ) +
    scale_fill_manual(
        values = c("Downregulating" = "#e31a1c"), # Green for Enhancer, Red for Silencing
        labels = c("Downregulating" = "Silencer"), # Rename labels
        name = "Regulation"
    ) +
    scale_size_continuous(
        range = c(2, 5), # Adjust dot size range for better visibility
        name = "Significance (-log10 p-value)"
    ) +
    guides(
        fill = guide_legend(override.aes = list(size = 6, shape = 21)) # Increase legend dot size
    ) +
    scale_x_continuous(limits = x_limits) +
    coord_cartesian(clip = "off") # Ensure no points are cut off
dev.off()

Warning message:
“No shared levels found between `names(values)` of the manual scale and the
data's fill values.”
Warning message:
“No shared levels found between `names(values)` of the manual scale and the
data's fill values.”


agg_record_409329079 
                   2

Warning message:
“No shared levels found between `names(values)` of the manual scale and the
data's fill values.”
Warning message:
“No shared levels found between `names(values)` of the manual scale and the
data's fill values.”


agg_record_409329079 
                   2